In [1]:
# Lets import the needed libraries. There are some quick annotations for what the unusual ones are used for.

import csv
import pandas as pd
import numpy as np
import re
import os
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification # An NLP model we use for sentiment analysis, it is pretrained. Details are in README.
from huggingface_hub import login # Our login/API key for hugging face, not strictly needed but helps run smoother.
from dotenv import load_dotenv # Adding secret API keys.
from datetime import datetime
import statsmodels.api as sm # For our regression model.
import matplotlib.pyplot as plt
import sqlite3 # For the purpose of further demonstrating the skills learned in this model, I chose to use SQL here.
from sklearn.preprocessing import MinMaxScaler # For data normalisation.
import scipy.stats as stats


# Below we load our API keys from the .env. You will require your own API keys, this is for the purpose of hiding mine.
# In this script we only use a HuggingFace key, althought this is not strictly required for the code to run, it does make it run smoother.

load_dotenv()
token = os.getenv('HF_TOKEN')
login(token)

c:\Users\Gucci Radiation\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
# In this cell we quickly load our data. We combine the Guardian News data, Alphavantage data and the NewsAPI data into one dataframe (news) via concat.

News_api_data = pd.read_csv('../Data/Data_Pre_Sentiment/NewsAPI_data.csv',encoding='utf-8') # Has some ecoding issues earlier, so just ensuring it is utf-8.
guardian_data = pd.read_csv('../Data/Data_Pre_Sentiment/guardian_data.csv',encoding='utf-8')
alphavantage_data = pd.read_csv('../Data/Data_Pre_Sentiment/alphavantage_data.csv',encoding='utf-8')
guardian_data = guardian_data.rename(columns={'headline': 'title', 'body_text': 'content'}, inplace=True) # Quickly rename Guardian and alphavantage columns so it will merge properly.
alphavantage_data = alphavantage_data.rename(columns={'summary':'content'})
news = pd.concat([guardian_data,News_api_data,alphavantage_data])

# We also load the financial data as well.

brent = pd.read_csv('../Data/Data_Pre_Sentiment/Brent_data.csv',encoding='utf-8')
wti = pd.read_csv('../Data/Data_Pre_Sentiment/WTI_data.csv',encoding='utf-8')
ovx = pd.read_csv('../Data/Data_Pre_Sentiment/ovx_data.csv',encoding='utf-8')
vix = pd.read_csv('../Data/Data_Pre_Sentiment/vix_data.csv',encoding='utf-8')
sp500 = pd.read_csv('../Data/Data_Pre_Sentiment/S&P500_data.csv',encoding='utf-8')


In [3]:
# This cell runs our sentiment analysis models over our news data.
# We use the ProsusAI/finbert model.
# Quick to run, only taking 2 minute (on my laptop).

prosus_model= AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert') # Defining the model and tokenizer.
prosus_tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')

# We use batching to speed up the process, if you are on a powerfull computer, you can increase the batch_size to speed up the run time.
nlp1 = pipeline('text-classification', model=prosus_model, tokenizer=prosus_tokenizer, truncation=True, max_length=512,batch_size=32)

# Creating a function to run over the title and content columns, it returns out the sentiment scores.
def sentiment(texts):
    results1 = nlp1([i if isinstance(i, str) and i.strip() else ' ' for i in texts])
    return [ i['score'] if i['label'] == 'positive' else -i['score'] if i['label'] == 'negative' else 0 for i in results1] 

news['title_score_prosus'] = sentiment(news['title'])
news['content_score_prosus'] = sentiment(news['content'])
news['final_score_prosus'] = news[['title_score_prosus','content_score_prosus']].mean(axis=1) # We create an additional column for an average score (of tile and content).

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 40214.42it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Quickly saving the news and sentiment data so we dont have to rerun the above. 
# We load it again in the next cell.

# news.to_csv('../Data/Data_Post_Sentiment/important_one.csv', index=False, encoding='utf-8')

In [4]:
# Merging later on throws a date data type error, so we again ensure everything is the same date data type here.

news = pd.read_csv('../Data/Data_Post_Sentiment/important_one.csv', encoding='utf-8')
news['date'] = pd.to_datetime(news['date'], format='%Y%m%d')
brent['date'] = pd.to_datetime(brent['date'])
wti['date'] = pd.to_datetime(wti['date'])
ovx['date'] = pd.to_datetime(ovx['date'])
vix['date'] = pd.to_datetime(vix['date'])
sp500['date'] = pd.to_datetime(sp500['date'])

In [ ]:
# We create new dataframes with the average daily sentiment using the final average score, title score alone, and content score alone.
# We can compare the sentiment between titles and content this way.
# We get the mean, but standard deviation and count will also be usefull

sentiment_by_day_for_avg = pd.DataFrame(news.groupby('date')['final_score_prosus'].agg(['mean','std','count']))
sentiment_by_day_for_title = pd.DataFrame(news.groupby('date')['title_score_prosus'].agg(['mean','std','count']))
sentiment_by_day_for_content = pd.DataFrame(news.groupby('date')['content_score_prosus'].agg(['mean','std','count']))

sentiment_by_day_for_avg.reset_index(inplace=True) # Resetting the index.
sentiment_by_day_for_title.reset_index(inplace=True)
sentiment_by_day_for_content.reset_index(inplace=True)

sentiment_by_day_for_avg = sentiment_by_day_for_avg[sentiment_by_day_for_avg['count']>1]
sentiment_by_day_for_title = sentiment_by_day_for_title[sentiment_by_day_for_title['count']>1]
sentiment_by_day_for_content = sentiment_by_day_for_content[sentiment_by_day_for_content['count']>1]


sentiment_by_day_for_avg.to_csv('../Data/Data_Post_Sentiment/sentiment_by_day_for_avg.csv', index=False, encoding='utf-8')
sentiment_by_day_for_title.to_csv('../Data/Data_Post_Sentiment/sentiment_by_day_for_title.csv', index=False, encoding='utf-8')
sentiment_by_day_for_content.to_csv('../Data/Data_Post_Sentiment/sentiment_by_day_for_content.csv', index=False, encoding='utf-8')



In [7]:
# We merge the average sentiment scores with brent,wti,ovx and vix.

pre_complete1 = pd.merge(sentiment_by_day_for_avg, brent ,on='date', how='inner')
pre_complete2 = pd.merge(pre_complete1, wti,on='date', how='inner')
pre_complete3 = pd.merge(pre_complete2, vix,on='date', how='inner')
pre_complete3 = pre_complete3.rename(columns={'mean': 'sentiment_mean','std': 'sentiment_std','daily_avg_x': 'brent','daily_avg_y': 'wti','daily_avg': 'vix'})
pre_complete4 = pd.merge(pre_complete3, ovx,on='date', how='inner')
pre_complete5 = pd.merge(pre_complete4, sp500,on='date', how='inner')
complete = pre_complete5.rename(columns={'daily_avg_x':'ovx','daily_avg_y':'sp500'})



In [8]:
# While this section could be done in python, for the purpose of further demonstrating the skills learned in this model, I chose to use SQL here.

conn = sqlite3.connect('newsdata.db')
complete.to_sql('news_and_data',conn,if_exists='replace',index=False)
conn.commit()

# We calculate a moving average to make underlying trends easier to spot.
def movingaverage(column,new_column):
    conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                       AVG({column}) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
                       AS {new_column}
                       FROM news_and_data;''')
    conn.execute("DROP TABLE news_and_data;")
    conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
    conn.commit()
    
movingaverage('sentiment_mean','sentiment_mean_moving_average')
movingaverage('ovx','ovx_moving_average')
movingaverage('vix','vix_moving_average')
movingaverage('wti','wti_moving_average')
movingaverage('brent','brent_moving_average')

# We invert some columns so that scales can be presenting the same way.
def invert(column,new_column):
    conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                       -{column} AS {new_column}
                       FROM news_and_data;''')
    conn.execute("DROP TABLE news_and_data;")
    conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
    conn.commit()

invert('sentiment_mean_moving_average','sentiment_mean_moving_average_inversed')

# We lag some columns to see if previous day x predicts y
def lag(column,new_column):
     conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                        (LAG({column}) OVER (ORDER BY date)) AS {new_column}
                        FROM news_and_data;''')
     conn.execute("DROP TABLE news_and_data;")
     conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
     conn.commit()

lag('brent','lagged_brent')
lag('wti','lagged_wti')
lag('vix','lagged_vix')
lag('ovx','lagged_ovx')
lag('sentiment_mean','lagged_sentiment_mean')
lag('sp500','lagged_sp500')

# We take the absolute values.
def abs_value(column,new_column):
     conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                        ABS({column}) AS {new_column}
                        FROM news_and_data;''')
     conn.execute("DROP TABLE news_and_data;")
     conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
     conn.commit()

abs_value('sentiment_mean','sentiment_mean_abs')

# We calculate percentage change.
def percentage_change(column,new_column):
     conn.executescript(f'''CREATE TABLE temp AS SELECT *,
                        (((({column}) - (LAG({column}) OVER(ORDER BY date))) / (LAG({column}) OVER(ORDER BY date)))*100) AS {new_column}
                        FROM news_and_data;''')
     conn.execute("DROP TABLE news_and_data;")
     conn.execute("ALTER TABLE temp RENAME TO news_and_data;")
     conn.commit()
     
percentage_change('brent','brent_percent')
percentage_change('wti','wti_percent')
percentage_change('vix','vix_percent')
percentage_change('ovx','ovx_percent')
percentage_change('sp500','sp500_percent')

In [9]:
# We take everythign in the SQL table and make it into a pandas dataframe.

All_needed_data = pd.read_sql('''SELECT * FROM news_and_data''',conn)
All_needed_data['date'] = pd.to_datetime(All_needed_data['date'], format='ISO8601')
All_needed_data = All_needed_data[All_needed_data['count']>1] # Removing extreme low counts as they heavily skew results
All_needed_data.to_csv('../Data/Data_Post_Sentiment/All_needed_data.csv', index=False, encoding='utf-8')
